1. 문서의 내용으 읽는다
2. 문서를 쪼갠다.
  - 토큰 수 초과로 답변을 생성하지 못할 수 있고
  - 문서가 길면(Input) 답변 생성이 오래걸림
3. 쪼갠 문서를 임배딩 -> 백터 데이터베이스에 저장
4. 질문이 있을 때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [2]:
# %pip install --upgrade --quiet  docx2txt langchain-community


In [3]:
# 문서 쪼개기 Text Splitter

# Character Text Splitter : 구분자를 하나만 넣을 수 있다.
# Recursive Character Text Splitter : 구분자를 여러개 넣을 수 있다.

# %pip install -qU langchain-text-splitters







In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # 문서를 쪼갤 때 하나의 청크가 가질 수 있는 청크의 수
    chunk_size=1500,
    # 문서를 곂치는 설정
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [5]:
len(document_list)

193

In [6]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings


load_dotenv()

embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')


In [7]:
# %pip install langchain-chroma

In [17]:
from langchain_chroma import Chroma

# database = Chroma.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     persist_directory='./chroma',
#     collection_name='chroma-tax'
#     )

database = Chroma(
    collection_name='chroma-tax',
    embedding_function=embedding,
    persist_directory='./chroma'
)


In [9]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')



In [21]:
prompt = f"""
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question : {query}
"""

In [22]:
ai_message = llm.invoke(prompt)

ai_message.content




'연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 여러 가지 요소를 고려해야 합니다. \n\n1. **소득공제 항목 확인:** 근로소득에는 공제 가능한 항목들이 여러 가지 있습니다. 예를 들어, 근로소득공제, 국민연금, 건강보험, 고용보험 등의 항목을 공제해야 합니다. \n\n2. **근로소득세 계산:** \n   - 근로소득공제를 적용한 후 과세표준을 계산합니다.\n   - 과세표준에 기본 소득세율을 적용하여 세금을 계산합니다.\n\n3. **세액공제 적용:** 계산된 세액에서 추가로 적용 가능한 세액공제를 적용합니다. 예를 들어, 자녀세액공제, 의료비, 교육비 및 보험료에 대한 세액공제 등이 있습니다.\n\n해당 내용의 예시는 소득공제 및 세액공제의 복잡한 부분이 포함되지 않았으므로, 실제 납부해야 하는 세금은 개인별로 차이가 있을 수 있습니다. 보다 구체적인 소득세 계산을 위해서는 국세청의 연말정산 시스템이나 세무 전문가의 상담을 통해 정확한 금액을 확인하는 것이 좋습니다. \n\n추가로, 각 연도마다 세율 및 공제 항목이 달라질 수 있으므로 최신의 세법 기준을 참고하는 것이 중요합니다.'

In [13]:
# %pip install -U langchain langchain_classic langchainhub --quiet

In [23]:
from langchain_classic import hub

prompt = hub.pull('rlm/rag-prompt')


In [15]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [24]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt': prompt}
)

In [25]:
ai_message = qa_chain.invoke({'query': query})

In [26]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '소득세 계산에 필요한 구체적인 세율이나 공제 항목에 대한 정보가 제공되지 않았습니다. 따라서 연봉 5천만원인 직장인의 소득세를 정확히 계산할 수 없습니다. 소득세를 정확히 계산하려면 관련 법령에 따른 세율, 공제 금액 등을 참조해야 합니다.'}